# 🖥️ Procesadores del Lenguaje II
## Unidad 5 — Entornos en Tiempo de Ejecución
### 40 Ejercicios Resueltos — Nivel Introductorio

---

**Bibliografía principal:**  
Aho, A. V., Sethi, R., & Lam, M. S. (2011). *Compiladores*. Pearson Educación de México SA de CV.

---

### 📋 Índice de temas

| Sección | Tema | Ejercicios |
|---------|------|------------|
| 1 | Introducción a los Entornos en Tiempo de Ejecución | 1–5 |
| 2 | Organización de Almacenamiento | 6–12 |
| 3 | Asignación en Pila y Registros de Activación | 13–22 |
| 4 | Acceso a Datos No Locales | 23–28 |
| 5 | Asignación en el Heap y Recolección de Basura | 29–34 |
| 6 | Paso de Parámetros | 35–40 |

---
## 🔹 Sección 1: Introducción a los Entornos en Tiempo de Ejecución
---

### Ejercicio 1 — Concepto de entorno en tiempo de ejecución

**Enunciado:** Define con tus propias palabras qué es un *entorno en tiempo de ejecución* y menciona tres responsabilidades que tiene el compilador con respecto a él.

**✅ Solución:**

Un **entorno en tiempo de ejecución** es el sistema de soporte —provisto por el compilador y el sistema operativo— que permite a un programa ejecutarse correctamente sobre una máquina física. No se trata solo del código generado, sino de la infraestructura que gestiona la memoria, el flujo de control entre procedimientos y el acceso a los datos.

Tres responsabilidades del compilador:

1. **Gestión de memoria:** Decidir cómo y dónde se asigna y libera la memoria para variables locales, globales y datos dinámicos.
2. **Paso de parámetros:** Generar el código que transfiere argumentos de una función a otra, según la convención elegida (valor, referencia, etc.).
3. **Acceso a variables no locales:** Crear mecanismos (como enlaces de acceso) para que un procedimiento pueda acceder a variables declaradas en ámbitos que lo contienen léxicamente.

### Ejercicio 2 — Rol del compilador vs. sistema operativo

**Enunciado:** Diferencia el papel del compilador y el del sistema operativo en la construcción del entorno en tiempo de ejecución.

**✅ Solución:**

| Aspecto | Compilador | Sistema Operativo |
|---------|-----------|-------------------|
| **Qué genera** | Código objeto + estrategia de gestión de memoria | Espacio de direcciones virtual para el proceso |
| **Memoria** | Decide la *disposición* (layout) de las regiones | *Asigna* físicamente páginas de memoria al proceso |
| **Llamadas** | Genera las secuencias de llamada/retorno | Gestiona el cambio de contexto entre procesos |
| **Heap** | Genera llamadas a `malloc`/`new`/GC | Provee la llamada al sistema (`brk`, `mmap`) |

En resumen: el compilador diseña la **estrategia** y el sistema operativo provee los **recursos físicos**.

### Ejercicio 3 — ¿Por qué el entorno de ejecución importa?

**Enunciado:** Menciona dos problemas concretos que pueden surgir si el entorno en tiempo de ejecución está mal diseñado.

**✅ Solución:**

1. **Desbordamiento de búfer (Buffer Overflow):** Si el compilador no reserva o valida correctamente el espacio en la pila, un dato puede escribirse más allá de su región asignada, corrompiendo datos adyacentes o permitiendo ataques de seguridad.

2. **Comportamiento incorrecto en recursividad:** Si el entorno no gestiona correctamente los registros de activación (por ejemplo, usando asignación estática para todos los datos, como en el Fortran antiguo), la recursividad no es posible y cualquier función que se llame a sí misma producirá resultados incorrectos.

### Ejercicio 4 — Clasificación de características del lenguaje

**Enunciado:** Para cada característica del lenguaje, indica qué componente del entorno de ejecución la hace posible.

| Característica | Componente |
|---------------|------------|
| Recursividad | ? |
| Variables dinámicas (`new`/`malloc`) | ? |
| Variables globales | ? |
| Funciones anidadas con acceso al ámbito exterior | ? |

**✅ Solución:**

| Característica | Componente |
|---------------|------------|
| Recursividad | Asignación en **pila** (cada llamada crea su propio registro de activación) |
| Variables dinámicas (`new`/`malloc`) | Asignación en el **heap** (montículo) |
| Variables globales | **Datos estáticos** (área de datos fija en tiempo de compilación) |
| Funciones anidadas con acceso al ámbito exterior | **Enlaces de acceso** (o displays) en los registros de activación |

### Ejercicio 5 — Simulación conceptual en Python

**Enunciado:** Escribe un pequeño programa Python que demuestre la idea de que cada llamada a función tiene su propio entorno (variables locales independientes), incluso en recursividad.

In [19]:
# ✅ Solución
# Cada llamada recursiva tiene su PROPIA variable local 'n'
# Esto es posible gracias a que cada activación tiene su
# registro de activación independiente en la pila.

def factorial(n):
    print(f"  Entrando a factorial({n}) — id de objeto 'n': {id(n)}")
    if n <= 1:
        resultado = 1
    else:
        resultado = n * factorial(n - 1)
    print(f"  Saliendo de factorial({n}) → resultado = {resultado}")
    return resultado

print("Calculando factorial(4):")
print(f"\nResultado final: {factorial(4)}")
print("\n⚡ Cada nivel de recursión tiene su propia copia de 'n',")
print("   gracias a la asignación en pila del registro de activación.")

Calculando factorial(4):
  Entrando a factorial(4) — id de objeto 'n': 11654472
  Entrando a factorial(3) — id de objeto 'n': 11654440
  Entrando a factorial(2) — id de objeto 'n': 11654408
  Entrando a factorial(1) — id de objeto 'n': 11654376
  Saliendo de factorial(1) → resultado = 1
  Saliendo de factorial(2) → resultado = 2
  Saliendo de factorial(3) → resultado = 6
  Saliendo de factorial(4) → resultado = 24

Resultado final: 24

⚡ Cada nivel de recursión tiene su propia copia de 'n',
   gracias a la asignación en pila del registro de activación.


---
## 🔹 Sección 2: Organización de Almacenamiento
---

### Ejercicio 6 — Regiones de memoria

**Enunciado:** Dibuja (en texto/ASCII) y describe las cuatro áreas principales en que se divide la memoria de un proceso en tiempo de ejecución.

**✅ Solución:**

```
Dirección alta
┌─────────────────────────┐
│         PILA            │  ← crece hacia abajo ↓
│   (Stack / automática)  │
├─────────────────────────┤
│          ...            │  (espacio libre entre pila y heap)
├─────────────────────────┤
│         HEAP            │  ← crece hacia arriba ↑
│   (montículo / dinámica)│
├─────────────────────────┤
│    DATOS ESTÁTICOS      │  (variables globales, constantes)
│    (tamaño fijo)        │
├─────────────────────────┤
│    CÓDIGO / TEXTO       │  (instrucciones de máquina, solo lectura)
└─────────────────────────┘
Dirección baja
```

| Región | Contenido | Ciclo de vida |
|--------|-----------|---------------|
| **Código** | Instrucciones de máquina | Toda la ejecución |
| **Datos estáticos** | Variables globales, constantes, strings literales | Toda la ejecución |
| **Pila** | Registros de activación de funciones | Duración de la función |
| **Heap** | Objetos dinámicos (`new`/`malloc`) | Hasta ser liberados |

### Ejercicio 7 — Clasificación de variables

**Enunciado:** Dado el siguiente programa C, indica en qué región de memoria se almacena cada variable.

```c
int contador = 0;          // (A)
char *mensaje = "Hola";    // (B) el literal

void incrementar(int n) {  // (C) parámetro n
    static int veces = 0;  // (D)
    int temp;              // (E)
    int *p = malloc(4);    // (F) el bloque asignado
    contador += n;
    veces++;
}
```

**✅ Solución:**

| Variable | Región | Justificación |
|----------|--------|---------------|
| (A) `contador` | **Datos estáticos** | Variable global, vive durante toda la ejecución |
| (B) `"Hola"` (literal) | **Datos estáticos** (solo lectura) | El literal de cadena es una constante del programa |
| (C) `n` (parámetro) | **Pila** | Parte del registro de activación de `incrementar` |
| (D) `veces` (static) | **Datos estáticos** | `static` dentro de función → vive toda la ejecución |
| (E) `temp` | **Pila** | Variable local automática |
| (F) bloque de `malloc` | **Heap** | Asignación dinámica explícita |

### Ejercicio 8 — Estrategias de asignación

**Enunciado:** Completa la siguiente tabla comparando las tres estrategias de asignación de almacenamiento.

**✅ Solución:**

| Criterio | Estática | Pila (automática) | Heap (dinámica) |
|----------|----------|-------------------|-----------------|
| **¿Cuándo se asigna?** | Tiempo de compilación | Al llamar a la función | En tiempo de ejecución (explícito) |
| **¿Cuándo se libera?** | Fin del programa | Al retornar la función | Cuando el programador/GC lo indique |
| **Velocidad** | Muy rápida (sin coste) | Rápida (solo mover SP) | Lenta (buscar bloque libre) |
| **Permite recursividad** | ❌ No | ✅ Sí | ✅ Sí |
| **Tamaño conocido en compilación** | ✅ Sí | ✅ Sí (o aproximado) | ❌ No necesariamente |
| **Ejemplo** | Variables globales | Variables locales | `new T()`, `malloc()` |

### Ejercicio 9 — Por qué Fortran antiguo no soportaba recursividad

**Enunciado:** Explica por qué las primeras versiones de Fortran no podían soportar funciones recursivas.

**✅ Solución:**

Las primeras versiones de Fortran utilizaban **asignación estática** para **todos** sus datos, incluidas las variables locales de subrutinas.

Esto significa que cada subrutina tenía una única ubicación de memoria fija para sus variables locales, establecida en tiempo de compilación. Si una función se llamaba a sí misma de forma recursiva, la segunda llamada **sobreescribiría** los datos de la primera llamada en esa ubicación estática, corrompiendo los resultados.

La recursividad requiere que cada activación tenga su **propio conjunto privado de variables locales**, lo cual solo es posible con la asignación en pila, donde cada llamada crea un nuevo registro de activación independiente.

### Ejercicio 10 — Simulación del crecimiento de la pila

**Enunciado:** Simula en Python cómo crece y decrece la pila de llamadas al ejecutar `f(3)` donde `f` es una función recursiva que llama a `f(n-1)`.

In [20]:
# ✅ Solución: Simulamos explícitamente la pila de llamadas

pila_llamadas = []  # representa la pila de ejecución

def f(n):
    # --- PRÓLOGO: push del registro de activación ---
    pila_llamadas.append(f"AR[f(n={n})]")
    print(f"PUSH  → Pila: {pila_llamadas}")

    if n > 0:
        f(n - 1)  # llamada recursiva

    # --- EPÍLOGO: pop del registro de activación ---
    pila_llamadas.pop()
    print(f"POP   → Pila: {pila_llamadas}")

print("=== Ejecutando f(3) ===")
f(3)
print("\n✅ La pila queda vacía al terminar, tal como en la ejecución real.")

=== Ejecutando f(3) ===
PUSH  → Pila: ['AR[f(n=3)]']
PUSH  → Pila: ['AR[f(n=3)]', 'AR[f(n=2)]']
PUSH  → Pila: ['AR[f(n=3)]', 'AR[f(n=2)]', 'AR[f(n=1)]']
PUSH  → Pila: ['AR[f(n=3)]', 'AR[f(n=2)]', 'AR[f(n=1)]', 'AR[f(n=0)]']
POP   → Pila: ['AR[f(n=3)]', 'AR[f(n=2)]', 'AR[f(n=1)]']
POP   → Pila: ['AR[f(n=3)]', 'AR[f(n=2)]']
POP   → Pila: ['AR[f(n=3)]']
POP   → Pila: []

✅ La pila queda vacía al terminar, tal como en la ejecución real.


### Ejercicio 11 — Disciplina LIFO de la pila

**Enunciado:** ¿Por qué la disciplina LIFO (Last-In, First-Out) es natural para gestionar las llamadas a procedimientos? Argumenta con un ejemplo.

**✅ Solución:**

La disciplina LIFO es natural porque **la función que se llama más recientemente siempre termina antes que la que la llamó**. Las activaciones están *anidadas en el tiempo*:

```
main()  →  llama a  A()  →  llama a  B()

Orden de inicio:  main → A → B
Orden de fin:     B termina, luego A, luego main
```

En la pila:
```
Después de llamar a B:
  [ AR(main) | AR(A) | AR(B) ]  ← cima

Al retornar B:
  [ AR(main) | AR(A) ]          ← volvemos al estado anterior
```

La función que **entró última** (B) es la que **sale primera**, perfectamente coincidente con la semántica LIFO. Esto hace que liberar el registro de activación sea tan simple como decrementar el puntero de pila (SP), sin necesidad de búsquedas.

### Ejercicio 12 — Fragmentación en el heap vs. la pila

**Enunciado:** Explica por qué la pila NO sufre fragmentación externa, mientras que el heap SÍ puede sufrirla.

**✅ Solución:**

**La pila no sufre fragmentación externa** porque solo se puede asignar y liberar memoria desde un extremo (la cima). Los registros de activación se apilan y desapilan en orden estricto LIFO. No existe la posibilidad de que queden "huecos" en medio de la pila, ya que cuando se libera un registro, siempre es el que está en la cima.

**El heap sí sufre fragmentación externa** porque la asignación y liberación de bloques puede ocurrir en **cualquier orden y cualquier tamaño**. Por ejemplo:

```
Estado inicial:  [LIBRE: 100 bytes]

malloc(30) → A:  [A:30 | LIBRE:70]
malloc(20) → B:  [A:30 | B:20 | LIBRE:50]
free(A):         [LIBRE:30 | B:20 | LIBRE:50]

malloc(60) → FALLA aunque hay 80 bytes libres,
             pero están en dos fragmentos no contiguos.
```

Este problema no puede ocurrir en la pila, donde la liberación siempre es contigua y desde la cima.

---
## 🔹 Sección 3: Asignación en Pila y Registros de Activación
---

### Ejercicio 13 — Estructura del registro de activación

**Enunciado:** Lista y describe brevemente los siete campos típicos de un registro de activación (activation record / stack frame).

**✅ Solución:**

```
┌──────────────────────────────┐  ← SP (Stack Pointer, cima)
│  7. Temporales               │  Valores intermedios de expresiones
├──────────────────────────────┤
│  6. Variables locales        │  Variables declaradas en el procedimiento
├──────────────────────────────┤
│  5. Estado de máquina guardado│  Registros de CPU del llamador
├──────────────────────────────┤  ← FP (Frame Pointer)
│  4. Enlace de control        │  Puntero al registro del llamador (FP anterior)
├──────────────────────────────┤
│  3. Enlace de acceso         │  Puntero al ámbito léxico padre
├──────────────────────────────┤
│  2. Dirección de retorno     │  A dónde saltar al terminar
├──────────────────────────────┤
│  1. Valor de retorno         │  Espacio para el resultado de la función
├──────────────────────────────┤
│  0. Parámetros reales        │  Argumentos de la llamada
└──────────────────────────────┘
```

Los campos no siguen un orden universal; varían según la arquitectura y la convención de llamada del compilador.

### Ejercicio 14 — SP vs FP

**Enunciado:** ¿Cuál es la diferencia entre el puntero de pila (SP) y el puntero de marco (FP)? ¿Por qué se necesitan ambos?

**✅ Solución:**

| Puntero | Apunta a | ¿Se mueve durante la función? |
|---------|----------|-------------------------------|
| **SP** (Stack Pointer) | La **cima actual** de la pila | ✅ Sí, constantemente (al evaluar expresiones, llamar funciones, etc.) |
| **FP** (Frame Pointer) | Una posición **fija dentro del registro de activación actual** | ❌ No (permanece constante durante toda la ejecución de la función) |

**¿Por qué necesitamos ambos?**

- El **FP** proporciona una **base estable** para acceder a las variables locales y parámetros mediante desplazamientos constantes (offsets). Por ejemplo: `[FP - 4]` siempre es la variable local `x`, sin importar cuánto haya cambiado SP.
- El **SP** es necesario para gestionar la cima de la pila al hacer operaciones (como nuevas llamadas, guardar temporales, etc.).

Sin FP, el compilador no podría generar desplazamientos fijos para acceder a las variables locales, ya que SP cambia constantemente.

### Ejercicio 15 — Simulación de un registro de activación

**Enunciado:** Implementa una clase Python `RegistroActivacion` que represente la estructura básica de un frame de pila.

In [21]:
# ✅ Solución

class RegistroActivacion:
    """Representa un registro de activación (stack frame) simplificado."""

    def __init__(self, nombre_proc, dir_retorno, enlace_control=None, enlace_acceso=None):
        self.nombre_proc    = nombre_proc      # nombre del procedimiento
        self.dir_retorno    = dir_retorno      # dirección de retorno
        self.enlace_control = enlace_control   # puntero al AR del llamador
        self.enlace_acceso  = enlace_acceso    # puntero al AR del padre léxico
        self.parametros     = {}               # parámetros formales
        self.locales        = {}               # variables locales
        self.valor_retorno  = None             # resultado de la función

    def set_param(self, nombre, valor):
        self.parametros[nombre] = valor

    def set_local(self, nombre, valor):
        self.locales[nombre] = valor

    def __str__(self):
        return (f"┌── AR[{self.nombre_proc}] ──────────────\n"
                f"│ Dir. retorno   : {self.dir_retorno}\n"
                f"│ Enlace control : {self.enlace_control}\n"
                f"│ Parámetros     : {self.parametros}\n"
                f"│ Locales        : {self.locales}\n"
                f"│ Valor retorno  : {self.valor_retorno}\n"
                f"└─────────────────────────────────────")


# Demostración: main llama a suma(a=3, b=5)
ar_main = RegistroActivacion("main", dir_retorno="sistema_operativo")
ar_main.set_local("x", 10)

ar_suma = RegistroActivacion("suma", dir_retorno="main:linea_7", enlace_control=ar_main)
ar_suma.set_param("a", 3)
ar_suma.set_param("b", 5)
ar_suma.set_local("resultado", 8)
ar_suma.valor_retorno = 8

print("=== Pila de ejecución ===\n")
print(ar_suma)   # en la cima
print(ar_main)   # debajo

=== Pila de ejecución ===

┌── AR[suma] ──────────────
│ Dir. retorno   : main:linea_7
│ Enlace control : ┌── AR[main] ──────────────
│ Dir. retorno   : sistema_operativo
│ Enlace control : None
│ Parámetros     : {}
│ Locales        : {'x': 10}
│ Valor retorno  : None
└─────────────────────────────────────
│ Parámetros     : {'a': 3, 'b': 5}
│ Locales        : {'resultado': 8}
│ Valor retorno  : 8
└─────────────────────────────────────
┌── AR[main] ──────────────
│ Dir. retorno   : sistema_operativo
│ Enlace control : None
│ Parámetros     : {}
│ Locales        : {'x': 10}
│ Valor retorno  : None
└─────────────────────────────────────


### Ejercicio 16 — Secuencia de llamada paso a paso

**Enunciado:** Describe paso a paso qué sucede cuando `main` llama a `suma(3, 5)`. Divide las acciones en: acciones del **llamador** y acciones del **llamado (prólogo)**.

**✅ Solución:**

#### Acciones del LLAMADOR (`main`) — Antes de saltar:

1. Evalúa los argumentos: `3` y `5`.
2. Los coloca en el espacio de parámetros del nuevo registro de activación (o en registros de la CPU según la convención).
3. Guarda la **dirección de retorno** (la instrucción siguiente a la llamada en `main`).
4. Guarda el valor actual del **FP** (enlace de control) en el nuevo frame.
5. Salta al código de `suma`.

#### Acciones del LLAMADO (`suma`) — Prólogo:

1. **Establece FP** ← SP (fija el puntero de marco al inicio del nuevo frame).
2. **Decrementa SP** para reservar espacio para variables locales de `suma`.
3. Guarda cualquier registro de CPU que necesite usar y que deba preservar.

```
        Antes          Después de la llamada
SP →  [AR(main)]    [AR(main)]  ← enlace de control
                    [AR(suma)]  ← SP, FP apuntan aquí
```

### Ejercicio 17 — Secuencia de retorno paso a paso

**Enunciado:** Describe los pasos de la **secuencia de retorno** cuando `suma` termina y devuelve el control a `main`.

**✅ Solución:**

#### Epílogo del LLAMADO (`suma`):

1. Coloca el **valor de retorno** (8) en la ubicación designada.
2. Restaura los registros de CPU que había guardado.
3. Restaura **SP** ← FP (deshace el espacio de variables locales).
4. Restaura **FP** ← enlace de control almacenado (FP del llamador).
5. Salta a la **dirección de retorno** guardada.

#### Continuación del LLAMADOR (`main`):

6. Lee el valor de retorno del lugar convenido.
7. Continúa la ejecución en la instrucción siguiente a la llamada.

```
        Antes          Después del retorno
SP →  [AR(main)]    [AR(main)]  ← SP y FP vuelven a main
      [AR(suma)]    (liberado: SP decrementó)
```

### Ejercicio 18 — Simulación completa de pila con llamadas anidadas

**Enunciado:** Simula la pila de llamadas cuando se ejecuta `main → A() → B()`. Muestra el estado de la pila en cada punto importante.

In [22]:
# ✅ Solución: simulación de la pila de llamadas

class PilaEjecucion:
    def __init__(self):
        self._pila = []

    def push(self, nombre):
        self._pila.append(nombre)
        self._mostrar(f"LLAMADA a {nombre}")

    def pop(self):
        nombre = self._pila.pop()
        self._mostrar(f"RETORNO de {nombre}")

    def _mostrar(self, evento):
        frames = " | ".join(self._pila) if self._pila else "(vacía)"
        print(f"  [{evento:25s}] Pila: [{frames}]")


pila = PilaEjecucion()

def B(pila):
    pila.push("B")
    # B hace su trabajo...
    pila.pop()

def A(pila):
    pila.push("A")
    B(pila)    # A llama a B
    pila.pop()

def main(pila):
    pila.push("main")
    A(pila)    # main llama a A
    pila.pop()

print("=== Simulación de pila: main → A → B ===")
main(pila)
print("\n✅ La pila está vacía al final — gestión correcta.")

=== Simulación de pila: main → A → B ===
  [LLAMADA a main           ] Pila: [main]
  [LLAMADA a A              ] Pila: [main | A]
  [LLAMADA a B              ] Pila: [main | A | B]
  [RETORNO de B             ] Pila: [main | A]
  [RETORNO de A             ] Pila: [main]
  [RETORNO de main          ] Pila: [(vacía)]

✅ La pila está vacía al final — gestión correcta.


### Ejercicio 19 — Cálculo de offsets en un registro de activación

**Enunciado:** Dado el siguiente procedimiento C, calcula el offset (desplazamiento desde FP) de cada variable local asumiendo que `int` ocupa 4 bytes, `double` ocupa 8 bytes y que las variables se asignan en orden de declaración sin padding.

```c
void proc(int a, int b) {
    int  x;     // local 1
    int  y;     // local 2
    double z;   // local 3
}
```

In [23]:
# ✅ Solución: cálculo de offsets

variables = [
    ("a", "int",    4),   # parámetro
    ("b", "int",    4),   # parámetro
    ("x", "int",    4),   # local
    ("y", "int",    4),   # local
    ("z", "double", 8),   # local
]

print(f"{'Variable':<10} {'Tipo':<10} {'Tamaño':>8} {'Offset (desde FP)':>20}")
print("-" * 52)

offset = 0
for nombre, tipo, tamanio in variables:
    print(f"{nombre:<10} {tipo:<10} {tamanio:>8} bytes   FP + {offset:>4}")
    offset += tamanio

print("-" * 52)
print(f"{'Tamaño total del área de datos:':<42} {offset} bytes")
print()
print("📌 El compilador genera instrucciones como:")
print("   LOAD R0, 8(FP)   → acceder a 'x'")
print("   LOAD R1, 12(FP)  → acceder a 'y'")
print("   LOADD F0, 16(FP) → acceder a 'z' (double)")

Variable   Tipo         Tamaño    Offset (desde FP)
----------------------------------------------------
a          int               4 bytes   FP +    0
b          int               4 bytes   FP +    4
x          int               4 bytes   FP +    8
y          int               4 bytes   FP +   12
z          double            8 bytes   FP +   16
----------------------------------------------------
Tamaño total del área de datos:            24 bytes

📌 El compilador genera instrucciones como:
   LOAD R0, 8(FP)   → acceder a 'x'
   LOAD R1, 12(FP)  → acceder a 'y'
   LOADD F0, 16(FP) → acceder a 'z' (double)


### Ejercicio 20 — Alineación de memoria (padding)

**Enunciado:** Recalcula los offsets del ejercicio anterior considerando que `double` (8 bytes) debe estar en una dirección múltiplo de 8. ¿Cuántos bytes de padding se necesitan?

In [24]:
# ✅ Solución: con alineación

def alinear(offset, alineacion):
    """Calcula el próximo offset alineado a 'alineacion' bytes."""
    resto = offset % alineacion
    if resto == 0:
        return offset, 0
    padding = alineacion - resto
    return offset + padding, padding

variables = [
    ("a",       "int",    4, 4),
    ("b",       "int",    4, 4),
    ("x",       "int",    4, 4),
    ("y",       "int",    4, 4),
    ("z",       "double", 8, 8),  # requiere alineación a 8
]

print(f"{'Variable':<8} {'Tipo':<10} {'Padding':>8} {'Offset':>8}")
print("-" * 40)

offset = 0
for nombre, tipo, tam, align in variables:
    offset, padding = alinear(offset, align)
    print(f"{nombre:<8} {tipo:<10} {padding:>7} bytes   {offset:>5}")
    offset += tam

print("-" * 40)
print(f"Tamaño total (con padding): {offset} bytes")
print("\n💡 Sin padding sería 24 bytes, con padding es más (depende del orden).")

Variable Tipo        Padding   Offset
----------------------------------------
a        int              0 bytes       0
b        int              0 bytes       4
x        int              0 bytes       8
y        int              0 bytes      12
z        double           0 bytes      16
----------------------------------------
Tamaño total (con padding): 24 bytes

💡 Sin padding sería 24 bytes, con padding es más (depende del orden).


### Ejercicio 21 — Datos locales de longitud variable

**Enunciado:** Explica el problema que plantean los arreglos de longitud variable (VLA) en C (`int A[n]`) para la asignación en pila y cómo se resuelve.

**✅ Solución:**

**El problema:** El compilador no puede calcular el tamaño total del registro de activación en tiempo de compilación, ya que `n` solo se conoce en tiempo de ejecución. Por lo tanto, no puede asignar offsets fijos para todas las variables.

**La solución:**

Se divide el registro de activación en dos partes:

```
┌──────────────────────────────┐  ← SP (varía)
│  Arreglo A[n]  (tamaño: n*4) │  (parte variable, tamaño conocido solo en ejecución)
├──────────────────────────────┤
│  ptr_A → apunta a inicio A   │  (offset fijo desde FP)
│  otras variables fijas       │  (offset fijo desde FP)
├──────────────────────────────┤  ← FP (fijo)
│  enlace de control, retorno  │
└──────────────────────────────┘
```

1. La parte fija del registro (parámetros, locales de tamaño conocido, puntero al arreglo) se ubica con offsets constantes desde FP.
2. En el **prólogo**, se evalúa `n`, se decrementa SP en `n * sizeof(int)`, y se guarda en `ptr_A` la dirección de inicio del arreglo.
3. El acceso a `A[i]` se realiza como `*(ptr_A + i)`, usando el puntero guardado.

Así se mantiene la eficiencia de la pila aunque el tamaño sea dinámico.

### Ejercicio 22 — Traza de recursión con registros de activación

**Enunciado:** Para la función `fibonacci(3)`, dibuja el estado de la pila en el momento en que se evalúa `fibonacci(1)` dentro de `fibonacci(2)` dentro de `fibonacci(3)`.

In [25]:
# ✅ Solución: traza con profundidad máxima de la pila

import sys

profundidad = [0]
max_profundidad = [0]
historia = []

def fibonacci(n, nivel=0):
    indent = "  " * nivel
    historia.append(f"{indent}PUSH AR[fib(n={n})]")
    profundidad[0] += 1
    max_profundidad[0] = max(max_profundidad[0], profundidad[0])

    if n <= 1:
        resultado = n
    else:
        resultado = fibonacci(n - 1, nivel + 1) + fibonacci(n - 2, nivel + 1)

    profundidad[0] -= 1
    historia.append(f"{indent}POP  AR[fib(n={n})] → {resultado}")
    return resultado

resultado = fibonacci(4)
print("=== Traza de registros de activación para fibonacci(4) ===")
for linea in historia:
    print(linea)
print(f"\nResultado: fibonacci(4) = {resultado}")
print(f"Profundidad máxima de la pila: {max_profundidad[0]} registros activos simultáneamente")

=== Traza de registros de activación para fibonacci(4) ===
PUSH AR[fib(n=4)]
  PUSH AR[fib(n=3)]
    PUSH AR[fib(n=2)]
      PUSH AR[fib(n=1)]
      POP  AR[fib(n=1)] → 1
      PUSH AR[fib(n=0)]
      POP  AR[fib(n=0)] → 0
    POP  AR[fib(n=2)] → 1
    PUSH AR[fib(n=1)]
    POP  AR[fib(n=1)] → 1
  POP  AR[fib(n=3)] → 2
  PUSH AR[fib(n=2)]
    PUSH AR[fib(n=1)]
    POP  AR[fib(n=1)] → 1
    PUSH AR[fib(n=0)]
    POP  AR[fib(n=0)] → 0
  POP  AR[fib(n=2)] → 1
POP  AR[fib(n=4)] → 3

Resultado: fibonacci(4) = 3
Profundidad máxima de la pila: 4 registros activos simultáneamente


---
## 🔹 Sección 4: Acceso a Datos No Locales
---

### Ejercicio 23 — Profundidad de anidamiento

**Enunciado:** Dado el siguiente pseudocódigo con procedimientos anidados, asigna la profundidad de anidamiento a cada procedimiento y variable.

```pascal
program principal;          { profundidad 1 }
  var x: integer;           { ¿profundidad? }
  procedure A;              { ¿profundidad? }
    var y: integer;         { ¿profundidad? }
    procedure B;            { ¿profundidad? }
      var z: integer;       { ¿profundidad? }
    begin { B } end;
  begin { A } end;
begin { principal } end.
```

**✅ Solución:**

| Elemento | Profundidad | Justificación |
|----------|-------------|---------------|
| `principal` (programa principal) | **1** | Nivel global |
| `x` (variable de `principal`) | **1** | Declarada en el ámbito de profundidad 1 |
| `A` (procedimiento) | **2** | Declarado dentro de `principal` (prof. 1 + 1) |
| `y` (variable de `A`) | **2** | Declarada en el ámbito de profundidad 2 |
| `B` (procedimiento) | **3** | Declarado dentro de `A` (prof. 2 + 1) |
| `z` (variable de `B`) | **3** | Declarada en el ámbito de profundidad 3 |

**Regla:** Si `B` (profundidad 3) necesita acceder a `x` (profundidad 1), debe seguir **3 − 1 = 2 saltos** en la cadena de acceso.

### Ejercicio 24 — Enlace de control vs. enlace de acceso

**Enunciado:** ¿Cuál es la diferencia entre el enlace de control (dynamic link) y el enlace de acceso (static/access link)? ¿Por qué el enlace de control solo no es suficiente para resolver variables no locales?

**✅ Solución:**

| Aspecto | Enlace de control | Enlace de acceso |
|---------|------------------|------------------|
| **Apunta a** | AR del procedimiento que **llamó** (dinámico) | AR del procedimiento que **contiene léxicamente** (estático) |
| **Sigue el orden** | De llamada dinámica | De anidamiento estático en el código |
| **Uso** | Restaurar FP y SP al retornar | Localizar variables no locales |

**¿Por qué el enlace de control no es suficiente?**

Porque el orden dinámico de llamadas puede ser distinto al orden de anidamiento léxico. Ejemplo:

```pascal
procedure A;          { prof. 2 }
  var x: integer;
  procedure B;        { prof. 3, anidada en A }
  begin B end;
  procedure C;        { prof. 3, anidada en A }
  begin B end;        { C llama a B dinámicamente }
begin A end;
```

Si `C` llama a `B`, el **enlace de control** de `B` apunta a `C`. Pero `B` necesita acceder a `x` de `A`, no de `C`. El **enlace de acceso** de `B` apunta a `A`, que es su padre léxico, permitiendo el acceso correcto.

### Ejercicio 25 — Traza de la cadena de acceso

**Enunciado:** En la situación del ejercicio anterior (C llama a B, ambos dentro de A), dibuja la pila y muestra las cadenas de control y de acceso.

**✅ Solución:**

```
Pila (creciendo hacia arriba):

┌──────────────────┐  ← SP (cima)
│ AR[B]            │
│  enlace control → AR[C]   (C llamó a B dinámicamente)
│  enlace acceso  → AR[A]   (A contiene léxicamente a B)
├──────────────────┤
│ AR[C]            │
│  enlace control → AR[A]   (A llamó a C)
│  enlace acceso  → AR[A]   (A contiene léxicamente a C)
├──────────────────┤
│ AR[A]            │
│  enlace control → AR[main]
│  enlace acceso  → AR[main] (o global)
│  var x = ...    │
├──────────────────┤
│ AR[main]         │
└──────────────────┘

Cadena de CONTROL:  B → C → A → main  (orden de llamadas)
Cadena de ACCESO:   B → A → main      (orden léxico de B)
```

Para que `B` acceda a `x` de `A`: sigue el **enlace de acceso** (1 salto) → llega a `AR[A]` → accede a `x` con el offset correcto.

### Ejercicio 26 — Simulación de la cadena de acceso

**Enunciado:** Implementa en Python una simulación que muestre cómo un procedimiento anidado busca una variable no local siguiendo la cadena de acceso.

In [26]:
# ✅ Solución: simulación de cadena de acceso

class AR:
    """Registro de activación simplificado."""
    def __init__(self, nombre, profundidad, enlace_acceso=None):
        self.nombre        = nombre
        self.profundidad   = profundidad
        self.enlace_acceso = enlace_acceso
        self.variables     = {}

    def __repr__(self):
        return f"AR[{self.nombre}(prof={self.profundidad})]"


def buscar_variable_no_local(ar_actual, nombre_var, saltos_requeridos):
    """Sigue la cadena de acceso 'saltos_requeridos' veces y busca la variable."""
    ar = ar_actual
    print(f"\nBuscando '{nombre_var}' desde {ar} (necesita {saltos_requeridos} saltos)")

    for i in range(saltos_requeridos):
        print(f"  Salto {i+1}: {ar} → {ar.enlace_acceso}")
        ar = ar.enlace_acceso

    valor = ar.variables.get(nombre_var, "⚠️ NO ENCONTRADA")
    print(f"  Encontrada en {ar}: {nombre_var} = {valor}")
    return valor


# Construir la estructura: principal → A → B
ar_principal = AR("principal", 1)
ar_principal.variables["x"] = 42

ar_A = AR("A", 2, enlace_acceso=ar_principal)
ar_A.variables["y"] = 10

ar_B = AR("B", 3, enlace_acceso=ar_A)
ar_B.variables["z"] = 5

# B accede a: z (local, 0 saltos), y de A (1 salto), x de principal (2 saltos)
buscar_variable_no_local(ar_B, "z", saltos_requeridos=0)
buscar_variable_no_local(ar_B, "y", saltos_requeridos=1)
buscar_variable_no_local(ar_B, "x", saltos_requeridos=2)


Buscando 'z' desde AR[B(prof=3)] (necesita 0 saltos)
  Encontrada en AR[B(prof=3)]: z = 5

Buscando 'y' desde AR[B(prof=3)] (necesita 1 saltos)
  Salto 1: AR[B(prof=3)] → AR[A(prof=2)]
  Encontrada en AR[A(prof=2)]: y = 10

Buscando 'x' desde AR[B(prof=3)] (necesita 2 saltos)
  Salto 1: AR[B(prof=3)] → AR[A(prof=2)]
  Salto 2: AR[A(prof=2)] → AR[principal(prof=1)]
  Encontrada en AR[principal(prof=1)]: x = 42


42

### Ejercicio 27 — Display vs. enlaces de acceso

**Enunciado:** Describe la técnica de *display* y explica por qué es más eficiente que los enlaces de acceso para acceder a variables no locales.

**✅ Solución:**

**Display:** Es un arreglo global `D` donde `D[i]` siempre apunta al **registro de activación más reciente** de un procedimiento a profundidad `i`.

```
D[1] → AR[principal]   (profundidad 1)
D[2] → AR[A]           (profundidad 2)
D[3] → AR[B]           (profundidad 3, procedimiento actual)
```

**Para acceder a la variable `x` en la profundidad `j`:**
- Con **enlaces de acceso:** hay que seguir la cadena `d_actual - j` veces → `d_actual - j` indirecciones de memoria.
- Con **display:** una sola instrucción: `*(D[j] + offset_x)` → **siempre 1 indirección**, sin importar la diferencia de profundidad.

**Desventaja del display:** Al entrar y salir de procedimientos, hay que **actualizar** las entradas del display, lo cual introduce una pequeña sobrecarga en las secuencias de llamada/retorno. Sin embargo, para programas con muchos accesos a variables no locales, el display es considerablemente más rápido.

### Ejercicio 28 — Clausuras (closures)

**Enunciado:** ¿Qué es una clausura y por qué es necesaria cuando se pasa un procedimiento anidado como parámetro? Ilustra con un ejemplo en Python.

In [27]:
# ✅ Solución

# Una CLAUSURA es un par (código, entorno léxico).
# El entorno léxico captura las variables no locales del procedimiento anidado.

def crear_sumador(base):          # profundidad 1
    """Retorna una función que suma 'incremento' a 'base'."""

    def sumar(incremento):        # profundidad 2 — accede a 'base' del padre
        return base + incremento  # 'base' es variable NO LOCAL de sumar

    return sumar   # retornamos la función junto con su entorno (clausura)

# Creamos dos clausuras distintas:
suma_10 = crear_sumador(10)   # clausura: { código: sumar, entorno: base=10 }
suma_20 = crear_sumador(20)   # clausura: { código: sumar, entorno: base=20 }

# Aunque crear_sumador ya retornó, las clausuras mantienen 'base' accesible
print(f"suma_10(5) = {suma_10(5)}")   # → 15
print(f"suma_20(5) = {suma_20(5)}")   # → 25
print()

# Verificamos que las clausuras tienen entornos distintos:
import inspect
print(f"Entorno de suma_10: base = {suma_10.__closure__[0].cell_contents}")
print(f"Entorno de suma_20: base = {suma_20.__closure__[0].cell_contents}")
print()
print("💡 Una clausura = puntero al código + puntero al entorno léxico (AR del padre).")
print("   Permite que la función use variables no locales incluso después de que")
print("   el procedimiento que las definió haya terminado.")

suma_10(5) = 15
suma_20(5) = 25

Entorno de suma_10: base = 10
Entorno de suma_20: base = 20

💡 Una clausura = puntero al código + puntero al entorno léxico (AR del padre).
   Permite que la función use variables no locales incluso después de que
   el procedimiento que las definió haya terminado.


---
## 🔹 Sección 5: Asignación en el Heap y Recolección de Basura
---

### Ejercicio 29 — Estrategias de búsqueda en el heap

**Enunciado:** Describe y compara las tres estrategias de búsqueda de bloque libre en el heap: First-fit, Best-fit y Next-fit.

**✅ Solución:**

| Estrategia | Descripción | Ventaja | Desventaja |
|-----------|-------------|---------|------------|
| **First-fit** | Elige el **primer** bloque libre de tamaño ≥ solicitado | Rápido | Fragmenta el inicio del heap |
| **Best-fit** | Elige el bloque libre **más pequeño** que sea ≥ solicitado | Mínimo desperdicio inmediato | Lento (recorre toda la lista); genera muchos fragmentos pequeños |  
| **Next-fit** | Como first-fit, pero **continúa desde donde terminó la última búsqueda** | Distribución uniforme | Puede fragmentar el heap más uniformemente |  

**En la práctica:** First-fit y Next-fit suelen tener mejor rendimiento real que Best-fit, a pesar de la intuición. Best-fit tiende a generar muchos fragmentos muy pequeños e inutilizables (fragmentación interna y externa combinadas).

### Ejercicio 30 — Simulación del administrador de heap (First-fit)

**Enunciado:** Implementa un administrador de heap simplificado con estrategia First-fit que soporte `malloc` y `free`.

In [28]:
# ✅ Solución: administrador de heap con first-fit

class AdministradorHeap:
    """Simula un heap con asignación First-fit."""

    def __init__(self, tamano_total):
        # Lista de bloques: (inicio, tamaño, libre/ocupado)
        self.bloques = [(0, tamano_total, True)]
        self.tamano  = tamano_total

    def malloc(self, tamano_solicitado):
        """First-fit: asigna el primer bloque libre suficientemente grande."""
        for i, (inicio, tam, libre) in enumerate(self.bloques):
            if libre and tam >= tamano_solicitado:
                # Dividir el bloque si sobra espacio
                self.bloques[i] = (inicio, tamano_solicitado, False)
                if tam > tamano_solicitado:
                    self.bloques.insert(i + 1, (inicio + tamano_solicitado,
                                                 tam - tamano_solicitado, True))
                print(f"malloc({tamano_solicitado}) → dir. {inicio}")
                return inicio
        print(f"malloc({tamano_solicitado}) → ❌ Sin memoria suficiente")
        return None

    def free(self, direccion):
        """Libera el bloque en la dirección dada y coalesca con vecinos libres."""
        for i, (inicio, tam, libre) in enumerate(self.bloques):
            if inicio == direccion and not libre:
                self.bloques[i] = (inicio, tam, True)
                self._coalescencia(i)
                print(f"free({direccion}) → bloque de {tam} bytes liberado")
                return
        print(f"free({direccion}) → ⚠️ Dirección no encontrada")

    def _coalescencia(self, i):
        """Fusiona bloques libres adyacentes."""
        # Fusionar con el siguiente si también está libre
        while i + 1 < len(self.bloques) and self.bloques[i + 1][2]:
            inicio, tam, _ = self.bloques[i]
            _, tam2, _     = self.bloques[i + 1]
            self.bloques[i] = (inicio, tam + tam2, True)
            del self.bloques[i + 1]

    def estado(self):
        print("  Estado del heap:", end=" ")
        for inicio, tam, libre in self.bloques:
            estado = "LIBRE" if libre else "OCUP."
            print(f"[{inicio}:{tam}b/{estado}]", end=" ")
        print()


# Demo
heap = AdministradorHeap(100)
heap.estado()
p1 = heap.malloc(30); heap.estado()
p2 = heap.malloc(20); heap.estado()
p3 = heap.malloc(40); heap.estado()
heap.free(p1);        heap.estado()  # quedan 30 bytes libres al inicio
heap.malloc(35)                      # no cabe (solo hay 30 al inicio, 10 al final)
heap.estado()
heap.free(p3);        heap.estado()  # coalescencia: 30+10=40 libres al inicio
heap.malloc(35);      heap.estado()  # ahora sí cabe

  Estado del heap: [0:100b/LIBRE] 
malloc(30) → dir. 0
  Estado del heap: [0:30b/OCUP.] [30:70b/LIBRE] 
malloc(20) → dir. 30
  Estado del heap: [0:30b/OCUP.] [30:20b/OCUP.] [50:50b/LIBRE] 
malloc(40) → dir. 50
  Estado del heap: [0:30b/OCUP.] [30:20b/OCUP.] [50:40b/OCUP.] [90:10b/LIBRE] 
free(0) → bloque de 30 bytes liberado
  Estado del heap: [0:30b/LIBRE] [30:20b/OCUP.] [50:40b/OCUP.] [90:10b/LIBRE] 
malloc(35) → ❌ Sin memoria suficiente
  Estado del heap: [0:30b/LIBRE] [30:20b/OCUP.] [50:40b/OCUP.] [90:10b/LIBRE] 
free(50) → bloque de 40 bytes liberado
  Estado del heap: [0:30b/LIBRE] [30:20b/OCUP.] [50:50b/LIBRE] 
malloc(35) → dir. 50
  Estado del heap: [0:30b/LIBRE] [30:20b/OCUP.] [50:35b/OCUP.] [85:15b/LIBRE] 


### Ejercicio 31 — Fugas de memoria y punteros colgantes

**Enunciado:** Distingue entre una *fuga de memoria* y un *puntero colgante* en C++. Da un ejemplo en pseudocódigo de cada uno.

**✅ Solución:**

#### Fuga de memoria (Memory Leak)
Memoria asignada que **nunca se libera**: el programa la mantiene como "en uso" aunque ya no tenga referencia a ella.

```cpp
// C++ — fuga de memoria
void funcion() {
    int* p = new int[100];   // asignamos 400 bytes en el heap
    // ... usamos p ...
    // ❌ olvidamos delete[] p;
}   // p se destruye (variable local), pero el heap NO se libera → FUGA
```

**Consecuencia:** Si esto ocurre en un bucle, el programa consume memoria hasta agotarla.

---

#### Puntero colgante (Dangling Pointer)
Un puntero que apunta a memoria **ya liberada**.

```cpp
// C++ — puntero colgante
int* p = new int(42);
int* q = p;              // q también apunta al mismo objeto
delete p;                // liberamos la memoria
// p y q ahora son punteros COLGANTES
*q = 99;                 // ❌ comportamiento indefinido: esa memoria pudo ser reasignada
```

**Consecuencia:** Corrupción de datos silenciosa o caída del programa.

### Ejercicio 32 — Algoritmo Mark-and-Sweep

**Enunciado:** Describe el algoritmo de recolección de basura *Mark-and-Sweep* y simúlalo con una pequeña red de objetos en Python.

In [29]:
# ✅ Solución: simulación de Mark-and-Sweep

class Objeto:
    def __init__(self, nombre):
        self.nombre  = nombre
        self.refs    = []    # referencias a otros objetos
        self.marcado = False

    def __repr__(self):
        return self.nombre


def marcar(obj):
    """Fase de marcado: DFS desde las raíces."""
    if obj.marcado:
        return
    obj.marcado = True
    print(f"  Marcando: {obj.nombre}")
    for ref in obj.refs:
        marcar(ref)


def barrer(todos_los_objetos):
    """Fase de barrido: liberar los no marcados."""
    vivos   = [o for o in todos_los_objetos if o.marcado]
    basura  = [o for o in todos_los_objetos if not o.marcado]
    print(f"  Liberando (basura): {[o.nombre for o in basura]}")
    # Desmarcar para la próxima ronda
    for o in vivos:
        o.marcado = False
    return vivos


# Crear objetos
A = Objeto("A")
B = Objeto("B")
C = Objeto("C")  # C solo es referenciado por B
D = Objeto("D")  # D no es alcanzable desde ninguna raíz → basura
E = Objeto("E")  # E tampoco es alcanzable → basura

# Establecer referencias
A.refs = [B]    # A → B
B.refs = [C]    # B → C
D.refs = [E]    # D → E (ciclo sin raíz)

todos = [A, B, C, D, E]
raices = [A]    # solo A es directamente accesible (ej: variable en la pila)

print("=== Fase de MARCADO ===")
for raiz in raices:
    marcar(raiz)

print("\n=== Fase de BARRIDO ===")
vivos = barrer(todos)

print(f"\nObjetos vivos tras la recolección: {vivos}")

=== Fase de MARCADO ===
  Marcando: A
  Marcando: B
  Marcando: C

=== Fase de BARRIDO ===
  Liberando (basura): ['D', 'E']

Objetos vivos tras la recolección: [A, B, C]


### Ejercicio 33 — Recolector por copia (Copying Collector)

**Enunciado:** Explica el funcionamiento del recolector de basura por copia y sus ventajas e inconvenientes respecto a Mark-and-Sweep.

**✅ Solución:**

#### Funcionamiento:

1. El heap se divide en **dos semiespacios** (`espacio-desde` y `espacio-hasta`), cada uno de la mitad del total.
2. Solo `espacio-desde` se usa normalmente.
3. Al activar la GC: el recolector **copia** todos los objetos alcanzables desde las raíces al `espacio-hasta`, en posiciones contiguas.
4. Los objetos copiados se referencian actualizando todos los punteros.
5. El `espacio-desde` completo se declara libre (no es necesario barrer objeto a objeto).
6. Los roles se **intercambian**: `espacio-hasta` pasa a ser el nuevo `espacio-desde`.

#### Comparación:

| Aspecto | Mark-and-Sweep | Copying Collector |
|---------|----------------|-------------------|
| **Fragmentación** | Puede sufrir fragmentación | ❌ Ninguna (compacta automáticamente) |
| **Memoria requerida** | Tamaño del heap | ✅/❌ El doble del heap real |
| **Velocidad (GC)** | Proporcional a todo el heap | Proporcional solo a objetos vivos |
| **Pausa del programa** | Sí (stop-the-world) | Sí (stop-the-world) |
| **Costo adicional** | Actualizar bits de marcado | Actualizar todos los punteros |
| **Asignación** | Necesita buscar en lista de libres | ✅ Muy rápido (solo mover puntero) |

### Ejercicio 34 — Fragmentación interna vs. externa

**Enunciado:** Distingue la fragmentación interna de la fragmentación externa y da un ejemplo de cada una en el contexto del heap.

In [30]:
# ✅ Solución: visualización de ambos tipos de fragmentación

def visualizar_heap(bloques, titulo):
    """Representa visualmente el heap como una línea de bloques."""
    print(f"\n{'='*50}")
    print(f"  {titulo}")
    print(f"{'='*50}")
    linea = ""
    for contenido, tam, nota in bloques:
        simbolo = contenido[0] if contenido != "LIBRE" else "·"
        celda = f"[{simbolo*tam}]"
        linea += celda
        if nota:
            print(f"  {celda:.<20} {nota}")
        else:
            print(f"  {celda}")
    print()

# === FRAGMENTACIÓN EXTERNA ===
# Hay 40 bytes libres en total, pero NO hay un bloque contiguo ≥ 30
visualizar_heap([
    ("LIBRE", 20, "← 20 bytes libres"),
    ("B_ocup",  15, "← ocupado (malloc 15)"),
    ("LIBRE", 20, "← 20 bytes libres"),
], "FRAGMENTACIÓN EXTERNA: 40 libres pero separados")

print("  malloc(30) → ❌ FALLA aunque hay 40 bytes libres totales")
print("  Causa: no hay un BLOQUE CONTIGUO de 30 bytes.\n")

# === FRAGMENTACIÓN INTERNA ===
# Se pidieron 14 bytes pero el asignador entregó 16 (alineación a 8)
visualizar_heap([
    ("A_obj",  14, "← bytes útiles del objeto"),
    ("PADD",   2,  "← 2 bytes de PADDING (desperdiciados dentro del bloque)"),
    ("LIBRE",  84, "← resto del heap libre"),
], "FRAGMENTACIÓN INTERNA: espacio perdido DENTRO de un bloque asignado")

print("  Se solicitaron 14 bytes, se asignaron 16 (alineación a 8 bytes).")
print("  Los 2 bytes de padding son fragmentación interna: inaccesibles al programa.")


  FRAGMENTACIÓN EXTERNA: 40 libres pero separados
  [····················] ← 20 bytes libres
  [BBBBBBBBBBBBBBB]... ← ocupado (malloc 15)
  [····················] ← 20 bytes libres

  malloc(30) → ❌ FALLA aunque hay 40 bytes libres totales
  Causa: no hay un BLOQUE CONTIGUO de 30 bytes.


  FRAGMENTACIÓN INTERNA: espacio perdido DENTRO de un bloque asignado
  [AAAAAAAAAAAAAA].... ← bytes útiles del objeto
  [PP]................ ← 2 bytes de PADDING (desperdiciados dentro del bloque)
  [····················································································] ← resto del heap libre

  Se solicitaron 14 bytes, se asignaron 16 (alineación a 8 bytes).
  Los 2 bytes de padding son fragmentación interna: inaccesibles al programa.


---
## 🔹 Sección 6: Paso de Parámetros
---

### Ejercicio 35 — Paso por valor: aislamiento entre funciones

**Enunciado:** Demuestra experimentalmente en Python que el paso por valor copia los datos y que la función llamada no puede modificar el original.

In [31]:
# ✅ Solución

def incrementar_por_valor(x):
    """Recibe una COPIA de x. Cualquier modificación es local."""
    print(f"  Dentro de la función: x = {x} (antes)")
    x = x + 100       # modificamos la copia local
    print(f"  Dentro de la función: x = {x} (después)")
    # El valor original NO cambia

valor_original = 5
print(f"Antes de llamar: valor_original = {valor_original}")
incrementar_por_valor(valor_original)
print(f"Después de llamar: valor_original = {valor_original}")
print()
print("✅ El valor original NO cambió → aislamiento del paso por valor.")
print("   En la pila: el parámetro 'x' es una variable DISTINTA con el mismo valor.")
print()
print("⚠️  NOTA: En Python, los tipos mutables (listas, dicts) se pasan por referencia")
print("    al objeto, no al contenedor. Modificar el CONTENIDO sí afecta el original.")

def modificar_lista(lst):
    lst.append(99)  # modifica el objeto original

mi_lista = [1, 2, 3]
modificar_lista(mi_lista)
print(f"\n  mi_lista después de modificar_lista(): {mi_lista}  ← sí cambió")

Antes de llamar: valor_original = 5
  Dentro de la función: x = 5 (antes)
  Dentro de la función: x = 105 (después)
Después de llamar: valor_original = 5

✅ El valor original NO cambió → aislamiento del paso por valor.
   En la pila: el parámetro 'x' es una variable DISTINTA con el mismo valor.

⚠️  NOTA: En Python, los tipos mutables (listas, dicts) se pasan por referencia
    al objeto, no al contenedor. Modificar el CONTENIDO sí afecta el original.

  mi_lista después de modificar_lista(): [1, 2, 3, 99]  ← sí cambió


### Ejercicio 36 — Paso por referencia: simulación con punteros

**Enunciado:** Simula el comportamiento del paso por referencia implementando una función `swap` que intercambie dos valores.

In [32]:
# ✅ Solución: simulamos paso por referencia con contenedores mutables

# --- Intento 1: swap por valor (NO funciona) ---
def swap_valor(a, b):
    """Solo intercambia copias locales."""
    a, b = b, a
    print(f"  Dentro de swap_valor: a={a}, b={b}")

x, y = 10, 20
print("=== Swap por VALOR (no funciona para el llamador) ===")
print(f"Antes: x={x}, y={y}")
swap_valor(x, y)
print(f"Después: x={x}, y={y}  ← SIN CAMBIO")

print()

# --- Intento 2: swap por referencia (con lista de un elemento como 'puntero') ---
def swap_ref(ref_a, ref_b):
    """Simula paso por referencia: modifica el contenido de los contenedores."""
    ref_a[0], ref_b[0] = ref_b[0], ref_a[0]
    print(f"  Dentro de swap_ref: ref_a[0]={ref_a[0]}, ref_b[0]={ref_b[0]}")

x_ref = [10]   # simulamos que pasamos la DIRECCIÓN de x
y_ref = [20]   # simulamos que pasamos la DIRECCIÓN de y

print("=== Swap por REFERENCIA (funciona para el llamador) ===")
print(f"Antes: x={x_ref[0]}, y={y_ref[0]}")
swap_ref(x_ref, y_ref)
print(f"Después: x={x_ref[0]}, y={y_ref[0]}  ← INTERCAMBIADOS ✅")
print()
print("En C/C++ esto se escribe como: void swap(int* a, int* b) { int t=*a; *a=*b; *b=t; }")

=== Swap por VALOR (no funciona para el llamador) ===
Antes: x=10, y=20
  Dentro de swap_valor: a=20, b=10
Después: x=10, y=20  ← SIN CAMBIO

=== Swap por REFERENCIA (funciona para el llamador) ===
Antes: x=10, y=20
  Dentro de swap_ref: ref_a[0]=20, ref_b[0]=10
Después: x=20, y=10  ← INTERCAMBIADOS ✅

En C/C++ esto se escribe como: void swap(int* a, int* b) { int t=*a; *a=*b; *b=t; }


### Ejercicio 37 — Costo de pasar estructuras grandes

**Enunciado:** Mide experimentalmente la diferencia de tiempo entre pasar una lista grande **por valor** (copia) vs. **por referencia** (puntero) en Python.

In [33]:
# ✅ Solución: comparación de rendimiento
import time

N = 10_000_000   # lista grande
REPETICIONES = 50

datos = list(range(N))  # lista de 10 millones de enteros

def procesar_por_copia(lista):
    """Recibe una COPIA de la lista (costoso)."""
    return len(lista)  # solo necesitamos el tamaño

def procesar_por_referencia(lista):
    """Recibe la misma referencia (barato)."""
    return len(lista)

# --- Medir paso por copia ---
t0 = time.perf_counter()
for _ in range(REPETICIONES):
    copia = datos.copy()   # simulamos la copia que haría el paso por valor
    procesar_por_copia(copia)
t_copia = time.perf_counter() - t0

# --- Medir paso por referencia ---
t0 = time.perf_counter()
for _ in range(REPETICIONES):
    procesar_por_referencia(datos)  # sin copia
t_ref = time.perf_counter() - t0

print(f"Lista de {N:,} elementos, {REPETICIONES} repeticiones")
print(f"Tiempo COPIA (por valor):     {t_copia:.4f} s")
print(f"Tiempo REFERENCIA:            {t_ref:.6f} s")
print(f"Ratio (copia/referencia):     {t_copia/t_ref:.0f}x más lento")
print()
print("💡 Conclusión: pasar estructuras grandes por valor tiene un costo")
print("   proporcional al tamaño. Por referencia solo se copia un puntero (fijo).")

Lista de 10,000,000 elementos, 50 repeticiones
Tiempo COPIA (por valor):     6.5500 s
Tiempo REFERENCIA:            0.000116 s
Ratio (copia/referencia):     56703x más lento

💡 Conclusión: pasar estructuras grandes por valor tiene un costo
   proporcional al tamaño. Por referencia solo se copia un puntero (fijo).


### Ejercicio 38 — Paso por copia-restauración

**Enunciado:** Explica el mecanismo de paso por copia-restauración (copy-restore) y muestra un caso donde su resultado difiere del paso por referencia verdadero.

In [34]:
# ✅ Solución: demostración de la diferencia entre copia-restauración y referencia

# Contexto: una variable global 'i' también es accesible dentro del procedimiento.
# Si pasamos 'i' por referencia, cualquier cambio a 'i' global
# se ve inmediatamente en el parámetro. Con copia-restauración, no.

print("=" * 50)
print("SIMULACIÓN: paso por REFERENCIA")
print("=" * 50)

i_ref = [1]   # simulamos i como variable global (paso por referencia)

def proc_referencia(param_ref, global_ref):
    # param_ref y global_ref apuntan al MISMO objeto (aliasing)
    param_ref[0] = param_ref[0] + 1    # param = 2
    global_ref[0] = global_ref[0] + 1  # i = 3 (¡ya era 2, sumamos 1 más!)
    # El cambio a global_ref SE VE a través de param_ref (mismo objeto)
    print(f"  Dentro: param={param_ref[0]}, i={global_ref[0]}")

i_ref[0] = 1
print(f"Antes: i={i_ref[0]}")
proc_referencia(i_ref, i_ref)  # pasamos i como parámetro Y como global → aliasing
print(f"Después (referencia): i={i_ref[0]}")

print()
print("=" * 50)
print("SIMULACIÓN: paso por COPIA-RESTAURACIÓN")
print("=" * 50)

i_val = 1   # variable global

def proc_copia_restauracion(param_inicial, global_dict):
    param = param_inicial          # 1. Copia al entrar
    param = param + 1              # opera: param = 2
    global_dict['i'] = global_dict['i'] + 1  # i = 2 (operación sobre global)
    print(f"  Dentro: param={param}, i={global_dict['i']}")
    global_dict['i'] = param       # 3. Restaura al salir: i ← param final = 2

g = {'i': 1}
print(f"Antes: i={g['i']}")
proc_copia_restauracion(g['i'], g)
print(f"Después (copia-restauración): i={g['i']}")

print()
print("💡 Con aliasing: referencia da 3, copia-restauración da 2.")
print("   La diferencia emerge porque la restauración sobreescribe")
print("   el cambio intermedio hecho a través del alias global.")

SIMULACIÓN: paso por REFERENCIA
Antes: i=1
  Dentro: param=3, i=3
Después (referencia): i=3

SIMULACIÓN: paso por COPIA-RESTAURACIÓN
Antes: i=1
  Dentro: param=2, i=2
Después (copia-restauración): i=2

💡 Con aliasing: referencia da 3, copia-restauración da 2.
   La diferencia emerge porque la restauración sobreescribe
   el cambio intermedio hecho a través del alias global.


### Ejercicio 39 — Paso por nombre (Call-by-Name)

**Enunciado:** Simula el paso por nombre usando funciones lambda (thunks). Muestra cómo el valor del argumento puede ser diferente en cada uso del parámetro.

In [35]:
# ✅ Solución: simulación de call-by-name con thunks

# En call-by-name, el argumento NO se evalúa al llamar.
# Se pasa un THUNK (función sin argumentos) que recalcula el valor cada vez.

estado = {'i': 0}  # variable externa que cambia durante la llamada

def doble_por_nombre(thunk_x):
    """Simula: procedure doble(x: expr) → return x + x
    Con call-by-name, 'x' se reevalúa CADA VEZ que se usa."""
    primera_evaluacion  = thunk_x()    # evalúa el argumento la 1ª vez
    estado['i'] += 1                   # el estado CAMBIA entre usos de x
    segunda_evaluacion  = thunk_x()    # evalúa el argumento la 2ª vez (¡puede cambiar!)
    print(f"  1ª evaluación de x: {primera_evaluacion}")
    print(f"  (i cambió a {estado['i']} entre evaluaciones)")
    print(f"  2ª evaluación de x: {segunda_evaluacion}")
    return primera_evaluacion + segunda_evaluacion

# El argumento es la expresión: i * 2  (depende del estado de i)
estado['i'] = 1
print("=== Paso por NOMBRE (call-by-name) ===")
print(f"i inicial = {estado['i']}")
resultado = doble_por_nombre(thunk_x=lambda: estado['i'] * 2)
print(f"Resultado de doble(i*2) = {resultado}")
print(f"(Esperaríamos 2*(1*2)=4 si fuera por valor, pero obtenemos {resultado})")
print()
print("💡 El thunk (lambda) captura la expresión i*2.")
print("   Cada uso del parámetro reevalúa la expresión en el contexto del LLAMADOR.")
print("   Esto hace call-by-name muy poderoso pero difícil de predecir.")

=== Paso por NOMBRE (call-by-name) ===
i inicial = 1
  1ª evaluación de x: 2
  (i cambió a 2 entre evaluaciones)
  2ª evaluación de x: 4
Resultado de doble(i*2) = 6
(Esperaríamos 2*(1*2)=4 si fuera por valor, pero obtenemos 6)

💡 El thunk (lambda) captura la expresión i*2.
   Cada uso del parámetro reevalúa la expresión en el contexto del LLAMADOR.
   Esto hace call-by-name muy poderoso pero difícil de predecir.


### Ejercicio 40 — Cuadro comparativo final y quiz de repaso

**Enunciado:** Completa el cuadro comparativo de todos los mecanismos de paso de parámetros y responde el mini-quiz de repaso de la unidad.

In [36]:
# ✅ Solución: cuadro comparativo interactivo

mecanismos = [
    {
        "nombre"        : "Paso por Valor",
        "qué se pasa"   : "Copia del valor del argumento",
        "llamado modifica original": "❌ No",
        "costo (struct)": "Alto (copia completa)",
        "aliasing"      : "No",
        "lenguajes"     : "C, Java (primitivos), Python (inmutables)",
    },
    {
        "nombre"        : "Paso por Referencia",
        "qué se pasa"   : "Dirección de memoria del argumento",
        "llamado modifica original": "✅ Sí",
        "costo (struct)": "Bajo (solo el puntero)",
        "aliasing"      : "Posible (peligroso)",
        "lenguajes"     : "C++ (&), Pascal (var), Python (mutables)",
    },
    {
        "nombre"        : "Paso por Copia-Restauración",
        "qué se pasa"   : "Copia; se restaura al retornar",
        "llamado modifica original": "✅ Al retornar",
        "costo (struct)": "Alto (2 copias)",
        "aliasing"      : "Distinto a referencia en alias",
        "lenguajes"     : "Ada (in out), FORTRAN antiguo",
    },
    {
        "nombre"        : "Paso por Nombre",
        "qué se pasa"   : "Thunk (expresión sin evaluar)",
        "llamado modifica original": "Depende (muy complejo)",
        "costo (struct)": "Muy alto (reevaluación repetida)",
        "aliasing"      : "Sí (y muy sorprendente)",
        "lenguajes"     : "Algol 60 (histórico), Haskell (lazy)",
    },
]

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║        COMPARATIVA DE MECANISMOS DE PASO DE PARÁMETROS              ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
for m in mecanismos:
    print(f"\n▶ {m['nombre']}")
    for k, v in m.items():
        if k != 'nombre':
            print(f"  {k:<35}: {v}")

print()
print("━" * 70)
print("MINI-QUIZ DE REPASO — Unidad 5")
print("━" * 70)

quiz = [
    ("¿Qué estructura sustenta las llamadas a procedimientos?",
     "La pila (stack) con registros de activación (AR)"),
    ("¿Qué puntero permanece fijo durante una llamada?",
     "El Frame Pointer (FP)"),
    ("¿Cuántos saltos en la cadena de acceso si la diferencia de profundidad es 3?",
     "3 saltos en la cadena de acceso"),
    ("¿Qué GC evita la fragmentación copiando objetos vivos?",
     "El recolector por copia (Copying Collector)"),
    ("¿Qué es una clausura?",
     "Un par (código del procedimiento, enlace al entorno léxico)"),
    ("¿Por qué la asignación estática impide la recursividad?",
     "Porque cada variable tiene una dirección fija única, imposibilitando múltiples activaciones"),
]

for i, (pregunta, respuesta) in enumerate(quiz, 1):
    print(f"\nP{i}: {pregunta}")
    print(f"  ✅ {respuesta}")

╔══════════════════════════════════════════════════════════════════════╗
║        COMPARATIVA DE MECANISMOS DE PASO DE PARÁMETROS              ║
╚══════════════════════════════════════════════════════════════════════╝

▶ Paso por Valor
  qué se pasa                        : Copia del valor del argumento
  llamado modifica original          : ❌ No
  costo (struct)                     : Alto (copia completa)
  aliasing                           : No
  lenguajes                          : C, Java (primitivos), Python (inmutables)

▶ Paso por Referencia
  qué se pasa                        : Dirección de memoria del argumento
  llamado modifica original          : ✅ Sí
  costo (struct)                     : Bajo (solo el puntero)
  aliasing                           : Posible (peligroso)
  lenguajes                          : C++ (&), Pascal (var), Python (mutables)

▶ Paso por Copia-Restauración
  qué se pasa                        : Copia; se restaura al retornar
  llamado modifica origi

---

## 🎓 Resumen de la Unidad 5

| Tema | Conceptos clave |
|------|-----------------|
| **Organización de memoria** | Código · Datos estáticos · Pila · Heap |
| **Estrategias de asignación** | Estática · En pila (LIFO) · En heap (dinámica) |
| **Registro de activación** | Variables locales · Parámetros · Enlace de control · Enlace de acceso · Dir. retorno · Valor retorno · Temporales |
| **Secuencias** | Prólogo (llamado) · Epílogo (llamado) · Pre-llamada (llamador) · Post-retorno (llamador) |
| **Acceso no local** | Profundidad de anidamiento · Cadena de acceso · Display |
| **Heap y GC** | First/Best/Next-fit · Mark-and-Sweep · Copia · Fragmentación |
| **Paso de parámetros** | Por valor · Por referencia · Copia-restauración · Por nombre · Clausuras |

---

**Bibliografía:**  
Aho, A. V., Sethi, R., & Lam, M. S. (2011). *Compiladores* (Cap. 7). Pearson Educación de México SA de CV.  
[Versión online](https://isergiobernalesgarcia.edu.pe/wp-content/uploads/2025/10/Compiladores-Alfred-V.-Aho-Monica-S.-Lam-Ravi-Sethi-Jeffrey-D.-Ullman.pdf)